# 第 2 章：语言模型的概率目标

这个 notebook 对应 `lessons/02_language_modeling.md`，演示 next-token prediction 的 input/label shift、bigram language model forward shape、单步训练和生成。

In [ ]:
import torch

from src.models.bigram_lm import (
    BigramLanguageModel,
    generate,
    make_lm_batch,
    train_step,
)

## 1. Input / Label Shift

语言模型训练时，input 是当前位置 token，label 是右移一位后的 next token。

In [ ]:
tokens = torch.tensor([[0, 1, 2, 3, 4]])
batch = make_lm_batch(tokens)

print("inputs:", batch.input_ids.tolist())
print("labels:", batch.labels.tolist())

## 2. Bigram LM Forward

Bigram 模型只根据当前 token 预测下一个 token，用它验证 `(B, T, V)` logits 契约。

In [ ]:
torch.manual_seed(0)
model = BigramLanguageModel(vocab_size=5, hidden_dim=8)
logits = model(batch.input_ids)

print(logits.shape)

## 3. 单步训练

训练步骤应计算 cross entropy、反传梯度，并更新 embedding 与 lm head 参数。

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)
before = model.token_embedding.weight.detach().clone()
loss = train_step(model, batch, optimizer)
after = model.token_embedding.weight.detach().clone()

print("loss:", round(loss, 4))
print("embedding_changed:", not torch.allclose(before, after))

## 4. 自回归生成

生成时每一步取最后位置 logits，采样一个新 token，再把它 append 回上下文。

In [ ]:
generated = generate(
    model,
    prompt=torch.tensor([[0, 1]]),
    max_new_tokens=4,
    temperature=1.0,
    top_k=3,
)

print(generated.tolist())